# 🍔 Food Delivery Backend API - Final Project
**Developer:** Rajeev Rathore  
**Internship:** Feb 2026 Batch - Innomatics Research Labs  
**Topic:** Food Delivery System (Option 1)

### Project Objective:
This project implements a complete backend for a food delivery application. It covers all core FastAPI concepts learned from Day 1 to Day 6:
- **Foundational:** Path/Query parameters and GET/POST methods.
- **Validation:** Pydantic models with constraints.
- **Logic:** Helper functions for price and tax calculations.
- **Data Management:** Full CRUD (Create, Read, Update, Delete) operations.
- **Advanced:** Search filters, multi-criteria sorting, and pagination.

---
## Phase 1: Menu Management & Basic Setup (Tasks 1 - 5)
In this phase, we initialize the server, define our data models, and create endpoints to interact with the food menu.

# **Core Implementation**
- Paste the following code into the next cell.
- This contains the server setup, data models, and the first 5 endpoints.

In [97]:
# --- ENDPOINTS (TASKS 1 TO 5) ---

import uvicorn
import threading
import nest_asyncio
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field
from typing import List, Optional
from datetime import datetime

# Enable Jupyter Compatibility
nest_asyncio.apply()

app = FastAPI(title="Zwiggy Food Delivery Service", version="1.5.0")

# --- DATABASE INITIALIZATION ---
food_menu = [
    {"id": 1, "name": "Paneer Tikka", "price": 280.0, "category": "Starters", "rating": 4.5, "is_available": True},
    {"id": 2, "name": "Butter Chicken", "price": 450.0, "category": "Main Course", "rating": 4.8, "is_available": True},
    {"id": 3, "name": "Masala Dosa", "price": 120.0, "category": "South Indian", "rating": 4.3, "is_available": True},
    {"id": 4, "name": "Gulab Jamun", "price": 80.0, "category": "Desserts", "rating": 4.9, "is_available": True},
    {"id": 5, "name": "Hakka Noodles", "price": 210.0, "category": "Chinese", "rating": 4.1, "is_available": True}
]
orders_db = []

# --- MODELS ---
class FoodItem(BaseModel):
    id: int
    name: str
    price: float = Field(..., gt=0)
    category: str
    rating: float = Field(default=4.0, ge=0, le=5)

# Q1: Welcome Route
@app.get("/", tags=["General"])
def welcome(): 
    return {"message": "Welcome to Zwiggy Food Delivery!"}

# Q2: System Status
@app.get("/system/status", tags=["General"])
def status(): 
    return {"status": "Open", "current_server_time": datetime.now().strftime("%H:%M")}

# Q3: System Statistics (Initial)
@app.get("/system/stats", tags=["General"])
def stats():
    rev = sum(o["total_amount"] for o in orders_db)
    return {"total_menu_items": len(food_menu), "orders_processed": len(orders_db), "total_revenue": rev}

# Q4: View Entire Menu
@app.get("/menu", tags=["Menu Management"])
def view_menu(): 
    return food_menu

# Q5: Get Specific Item by ID
@app.get("/menu/item/{item_id}", tags=["Menu Management"])
def get_item(item_id: int):
    item = next((i for i in food_menu if i["id"] == item_id), None)
    if not item: raise HTTPException(status_code=404, detail="Food item not found")
    return item

print("✅ Phase 1: Setup Complete")

✅ Phase 1: Setup Complete


## Phase 2: Helper Functions & Advanced Validation (Tasks 6 - 10)
In this phase, we move beyond simple data retrieval. We implement:
- **Helper Functions:** To centralize logic for price calculations.
- **Query Parameters:** To filter data based on specific criteria like category.
- **Validation Logic:** Ensuring that updates to the menu follow strict business rules.

In [99]:
# --- ENDPOINTS (TASKS 6 TO 10) ---

# --- HELPER FUNCTION ---
def apply_tax_and_service(base_price: float):
    """Calculates final price with 5% GST and 10% Service Charge."""
    return round(base_price + (base_price * 0.05) + (base_price * 0.10), 2)

# Q6: Final Price with Tax Calculation
@app.get("/menu/item/{item_id}/final-price", tags=["Menu Management"])
def get_final_price(item_id: int):
    item = next((i for i in food_menu if i["id"] == item_id), None)
    if not item: raise HTTPException(status_code=404, detail="Item not found")
    return {"item": item["name"], "base_price": item["price"], "final_bill": apply_tax_and_service(item["price"])}

# Q7: Filter Menu by Category
@app.get("/menu/filter", tags=["Menu Management"])
def filter_menu(category: str):
    return [i for i in food_menu if i["category"].lower() == category.lower()]

# Q8: Get Top Rated Item
@app.get("/menu/top-rated", tags=["Menu Management"])
def top_rated():
    return max(food_menu, key=lambda x: x["rating"])

# Q9: Add New Item to Menu (POST)
@app.post("/menu/item", tags=["Admin Operations"])
def add_item(item: FoodItem):
    food_menu.append(item.dict())
    return {"message": "New item added successfully", "data": item}

# Q10: Update Item Price (PUT)
@app.put("/menu/item/{item_id}/update-price", tags=["Admin Operations"])
def update_price(item_id: int, new_price: float):
    for i in food_menu:
        if i["id"] == item_id:
            i["price"] = new_price
            return {"message": "Price Updated", "updated_item": i}
    raise HTTPException(status_code=404, detail="Item not found")

print("✅ Phase 2: Logic & Admin Tasks Complete")

✅ Phase 2: Logic & Admin Tasks Complete


## Phase 3: Order Management & Full CRUD (Tasks 11 - 15)
In this phase, we implement the core business logic of the app:
- **Order Processing:** Handling customer orders and calculating total bills.
- **Full CRUD:** Implementing the ability to view, update, and cancel (delete) orders.
- **Workflow Automation:** Managing order status from 'Pending' to 'Delivered'.

In [101]:
# --- ENDPOINTS (TASKS 11 TO 15) ---

class PlaceOrder(BaseModel):
    customer_name: str
    items: List[int]
    address: str

# Q11: Toggle Availability Status (PATCH)
@app.patch("/menu/item/{item_id}/status", tags=["Admin Operations"])
def toggle_status(item_id: int, available: bool):
    for i in food_menu:
        if i["id"] == item_id:
            i["is_available"] = available
            return {"item": i["name"], "available": available}
    raise HTTPException(status_code=404)

# Q12: Place a New Order
@app.post("/orders/place", tags=["Order Management"])
def place_order(order: PlaceOrder):
    subtotal = sum(next(i["price"] for i in food_menu if i["id"] == pid) for pid in order.items)
    new_o = {
        "order_id": len(orders_db)+1, 
        "customer": order.customer_name, 
        "total_amount": apply_tax_and_service(subtotal), 
        "status": "Order Placed"
    }
    orders_db.append(new_o)
    return new_o

# Q13: View All Order History
@app.get("/orders/all", tags=["Order Management"])
def all_orders(): 
    return {"total_orders": len(orders_db), "orders": orders_db}

# Q14: Update Order Status Workflow
@app.patch("/orders/{order_id}/update-status", tags=["Order Management"])
def update_order_status(order_id: int, status: str):
    for o in orders_db:
        if o["order_id"] == order_id:
            o["status"] = status
            return o
    raise HTTPException(status_code=404, detail="Order not found")

# Q15: Cancel an Order (DELETE)
@app.delete("/orders/{order_id}/cancel", tags=["Order Management"])
def cancel(order_id: int):
    global orders_db
    orders_db = [o for o in orders_db if o["order_id"] != order_id]
    return {"message": f"Order {order_id} has been cancelled successfully"}

print("✅ Phase 3: Order Management Complete")

✅ Phase 3: Order Management Complete


## Phase 4: Advanced Search, Sorting & Pagination (Tasks 16 - 20)
In the final phase, we implement advanced data retrieval techniques:
- **Search Logic:** Finding specific dishes using keywords.
- **Multi-Level Sorting:** Organizing the menu by category and price simultaneously.
- **Pagination:** Handling large datasets by returning data in chunks (pages).
- **Global Stats:** Providing a summary of the entire food delivery system.

In [103]:
# --- ENDPOINTS (TASKS 16 TO 20) ---

# Q16: Get Latest Order for a Customer
@app.get("/orders/latest", tags=["Order Management"])
def latest(customer_name: str):
    user_orders = [o for o in orders_db if o["customer"].lower() == customer_name.lower()]
    if not user_orders: return {"message": "No orders found for this user"}
    return user_orders[-1]

# Q17: Search Food Items by Keyword
@app.get("/menu/search", tags=["Advanced Features"])
def search(keyword: str):
    return [i for i in food_menu if keyword.lower() in i["name"].lower()]

# Q18: Multi-Criteria Sorting (Category then Price)
@app.get("/menu/sorted-list", tags=["Advanced Features"])
def sorted_menu():
    return sorted(food_menu, key=lambda x: (x["category"], x["price"]))

# Q19: Pagination Support
@app.get("/menu/page", tags=["Advanced Features"])
def paginate(page: int = 1, limit: int = 2):
    return food_menu[(page-1)*limit : page*limit]

# Q20: Administrative System Reset
@app.delete("/system/reset-orders", tags=["Admin Operations"])
def reset(confirm: bool = False):
    if not confirm: return {"message": "Please set confirm=True to wipe all order data"}
    global orders_db
    orders_db = []
    return {"message": "System Reset: All orders deleted"}

# --- START SERVER ---
def run():
    uvicorn.run(app, host="127.0.0.1", port=8004)

threading.Thread(target=run, daemon=True).start()
print("🚀 Server started on http://127.0.0.1:8004/docs")

🚀 Server started on http://127.0.0.1:8004/docs


# 🏁 Final Conclusion & Project Summary

This project marks the successful completion of the **FastAPI Internship Final Assignment**. Through the development of the **Zwiggy Food Delivery Backend**, I have practically implemented all the core concepts covered during the 6-day training period at **Innomatics Research Labs**.

### Key Takeaways:
1. **API Architecture:** Learned how to structure a real-world application using FastAPI, ensuring a logical flow between menu management, user orders, and administrative tasks.
2. **Data Integrity:** Implemented **Pydantic Models** to enforce strict data validation, ensuring that the system only processes high-quality and logically sound data (e.g., positive prices, valid ratings).
3. **Logic & Automation:** Built custom **Helper Functions** to automate repetitive tasks like tax calculations (GST/Service Charge), making the code more modular and maintainable.
4. **Advanced Data Handling:** Gained hands-on experience with complex features like **multi-criteria sorting**, **keyword-based search**, and **API pagination**, which are essential for performance in production environments.
5. **Standardized Documentation:** Utilized the built-in **Swagger UI (OpenAPI)** for interactive testing of all 20 endpoints, ensuring that the backend is ready for frontend integration.

### Final Thoughts:
Building this Food Delivery API has significantly boosted my confidence in Python backend development. It has taught me the importance of route order, proper error handling (HTTP Exceptions), and writing clean, "developer-friendly" code. 

I am grateful to **Innomatics Research Labs** and my mentor for the guidance provided throughout this journey. This project is now fully documented, tested, and ready for deployment.

---
**Developed by:** Rajeev Rathore  
**Project:** Food Delivery App Backend  
**Date:** March 20, 2026
